# 1. Import Libraries and Create SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler #ML expects all feature in one vector
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
spark = SparkSession.builder.appName('Boston Housing Regression').master("local[*]").getOrCreate()

# 2. Load the DataSet

In [2]:
df = spark.read.csv("Boston.csv", header=True, inferSchema=True)
df.show(10)

+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|_c0|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio| black|lstat|medv|
+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|  1|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|  2|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|  3|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|  4|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|  5|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|  6|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|  7|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|  8|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|  9|0.21124|12.5| 7.

In [3]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- black: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



# 3. Select Features and Target

In [4]:
df.columns

['_c0',
 'crim',
 'zn',
 'indus',
 'chas',
 'nox',
 'rm',
 'age',
 'dis',
 'rad',
 'tax',
 'ptratio',
 'black',
 'lstat',
 'medv']

In [5]:
feature_columns = ["crim", "zn", "indus", "chas", "nox", "rm", "age", "dis","rad","tax", "ptratio", "black", "lstat"]

In [6]:
data = df.select(feature_columns + ["medv"])
data.show(5)

+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+
|   crim|  zn|indus|chas|  nox|   rm| age|   dis|rad|tax|ptratio| black|lstat|medv|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+
|0.00632|18.0| 2.31|   0|0.538|6.575|65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|0.02731| 0.0| 7.07|   0|0.469|6.421|78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|0.02729| 0.0| 7.07|   0|0.469|7.185|61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|0.03237| 0.0| 2.18|   0|0.458|6.998|45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|0.06905| 0.0| 2.18|   0|0.458|7.147|54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+
only showing top 5 rows


# 4. Convert Features into a Single Vector
* MLlib expects input features in one vector column



In [7]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
# 13 Feature columns are converted into as 1 feature column through VectorAssembler

In [8]:
data = assembler.transform(data)
data.show(5)

+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+--------------------+
|   crim|  zn|indus|chas|  nox|   rm| age|   dis|rad|tax|ptratio| black|lstat|medv|            features|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+--------------------+
|0.00632|18.0| 2.31|   0|0.538|6.575|65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|[0.00632,18.0,2.3...|
|0.02731| 0.0| 7.07|   0|0.469|6.421|78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|[0.02731,0.0,7.07...|
|0.02729| 0.0| 7.07|   0|0.469|7.185|61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|[0.02729,0.0,7.07...|
|0.03237| 0.0| 2.18|   0|0.458|6.998|45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|[0.03237,0.0,2.18...|
|0.06905| 0.0| 2.18|   0|0.458|7.147|54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|[0.06905,0.0,2.18...|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+------+-----+----+--------------------+
only showing top 5 rows


In [9]:
data.select("features", "medv").show(5, truncate = False)

+-------------------------------------------------------------------------+----+
|features                                                                 |medv|
+-------------------------------------------------------------------------+----+
|[0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.09,1.0,296.0,15.3,396.9,4.98]  |24.0|
|[0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.9,9.14] |21.6|
|[0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03]|34.7|
|[0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94]|33.4|
|[0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.9,5.33] |36.2|
+-------------------------------------------------------------------------+----+
only showing top 5 rows


# 5. Training and Testing
* 80% Training and 20% Testing

In [10]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
print("Training Rows : ", train_data.count())
print("Testing Rows : ",test_data.count())

Training Rows :  430
Testing Rows :  76


# 6. Create the Linear Regresssion

In [11]:
lr = LinearRegression(featuresCol="features", labelCol="medv")

# 7. Train the Model

In [12]:
model = lr.fit(train_data)
print("Model is Trained Successfully")

Model is Trained Successfully


# 8. Make Predictions

In [13]:
predictions = model.transform(test_data)
predictions.select("medv","prediction").show()

+----+------------------+
|medv|        prediction|
+----+------------------+
|22.0|27.482274018186136|
|50.0| 40.59821928572501|
|29.1|31.560171030407385|
|32.9| 30.50410754091398|
|42.3| 36.71084264945597|
|34.7|30.375442767948023|
|18.5| 19.59899004373776|
|34.9|34.091019647336594|
|23.5|30.499298725303998|
|27.9| 31.92001039736025|
|24.8| 26.09338517396815|
|20.7|  21.6855121924401|
|23.2| 27.35989331536614|
|28.0| 28.33678346582046|
|24.8| 30.92596058671465|
|22.6|27.029279789450975|
|22.5|22.283609778223276|
|22.4| 23.31506724074555|
|30.5|30.187595627825928|
|20.3| 22.68581977785559|
+----+------------------+
only showing top 20 rows


# 9. Evaluate the Model

In [14]:
evaluator_rmse = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="rmse")

evaluator_r2 = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("RMSE : ", rmse)
print("R2 : ", r2)

RMSE :  4.671806485171298
R2 :  0.7931522547835954


In [15]:
spark.stop()